# 単回帰分析

単回帰は、機械学習の中でもいちばん小さな予測モデルです。入力が 1 つ、出力も 1 つ、そしてその関係を 1 本の直線で近似します。モデルとしては単純ですが、予測、誤差、評価分割という機械学習の骨格がほぼ全部入っています。

このノートでは、まず直線でうまく説明できる小さなデータから始めて、係数を手で求め、誤差を見て、最後に評価データを分ける意味まで確認します。


## まずは、1 本の線でどこまで説明できるかを見る

単回帰の入口にある問いは単純です。観測点がいくつか並んでいるとき、それを 1 本の線でどこまでうまく説明できるか。まずはその感覚をつかむことが大切です。

傾きは「入力が 1 増えたときに予測がどれだけ増えるか」、切片は「入力が 0 のときの基準点」です。この 2 つで、最小の予測器ができます。


このノートでは、まず小さな観測点に線を引くところから始め、その線がどこで外れるかを見ます。そのあとで、線を少し賢くしたら何が良くなるか、そして評価用データを分けると印象がどう変わるかを追います。

つまり、単回帰の話でありながら、機械学習の流れ全体を縮小模型として体験する構成です。


## このノートで使う基本語彙

- 切片: `x=0` のときの予測値
- 傾き: `x` が 1 増えたときに予測がどれだけ変わるか
- 残差: 実測値と予測値の差
- MSE: 残差を二乗して平均した指標

残差は 1 件ごとの外れ方を見せ、MSE は全体としてどれくらい外しているかをまとめて見せます。両方を行き来すると、平均スコアだけでは見えない偏りに気づきやすくなります。


## 直線を作って、外れ方を見る

ここから先は 1 本の直線を少しずつ詳しく見ていきます。まずデータを置き、次に係数を求め、予測を出し、どこで外しているかを確認します。流れは単純ですが、この順序がそのまま後のモデルにも残ります。


## 指標は「何をどれだけ外したか」を要約したもの

このノートで見る `test MAE` は、評価データでの平均的な外れ幅です。小さいほど予測が近い、と読めば大丈夫です。

ただし、数値が 1 つ良いからといって安心せず、どの点で大きく外したのかを残差でも見るのが大切です。


## 先に構えたい誤解

単回帰では、平均や傾きを計算できるようになると「理解した気」になりやすいのですが、それだけでは不十分です。係数が求まることと、そのモデルが新しいデータでも当たることは別問題です。

このノートの後半で評価分割を入れるのは、その差をはっきり体験するためです。


## どこまでをこのノートで扱うか

ここでは、直線で説明する最小ケースに集中します。厳密な統計推定や信頼区間ではなく、「予測器としての回帰」を読むことが中心です。


## まずは、直線で説明できそうなデータを置く

最初に、`x` が増えると `y` もほぼ比例して増える小さなデータを作ります。ここではまず散らばった観測点ではなく、関係が見えやすい最小例を使います。


In [ ]:
x = [1, 2, 3, 4, 5]
y = [2.1, 4.2, 6.1, 8.2, 10.1]
pairs = list(zip(x, y))
print('task = simple-regression')
print('pairs =', pairs)

この 5 点を見ながら、「だいたいどんな直線が通りそうか」を頭の中で先に描いてみてください。あとで計算した傾きと比べると、数式が急に具体的になります。


## 平均と差分から、直線の係数を求める

最小二乗法と聞くと式が重そうに見えますが、実装としてやっていることは平均を取り、平均との差を作り、それを掛け合わせて足し上げるだけです。大事なのは、記号の形ではなく「どの向きに増える関係を拾っているか」です。


In [ ]:
x_bar = sum(x) / len(x)
y_bar = sum(y) / len(y)
num = sum((xi - x_bar) * (yi - y_bar) for xi, yi in pairs)
den = sum((xi - x_bar) ** 2 for xi in x)
w1 = num / den; w0 = y_bar - w1 * x_bar
print('w0, w1 =', round(w0, 4), round(w1, 4))

抽象的な式も、コードに落とすと平均、差分、和という部品に分かれます。回帰の式が怖く見えたら、まずこの粒度まで分解して読むのがいちばん安全です。


## 式は、予測と誤差を行き来するためにある

回帰式は、予測値を出すための形です。損失関数は、その予測がどれだけ外れているかを測るための形です。学習とは、この 2 つを往復しながら外れを小さくする作業だと見ると整理しやすくなります。


## 求めた直線で、どこを外しているかを見る

係数が出たら終わりではなく、そこから予測を作って残差を見ます。モデル改善は、平均スコアを眺めることより、どこでどう外しているかを観察するところから始まります。


In [ ]:
pred = [w0 + w1 * xi for xi in x]
residual = [yi - pi for yi, pi in zip(y, pred)]
mse = sum((r ** 2) for r in residual) / len(residual)
print('pred =', pred)
print('mse =', round(mse, 8))

平均誤差が小さくても、ある範囲でだけ一方向に外していれば、モデルの表現力が足りないかもしれません。残差を見るのは、その偏りを見つけるためです。


## 特徴量を増やすと、どこまで表現できるか

次は少しだけ表現力を増やしてみます。線だけでは足りない関係があるなら、特徴量の持ち方を変えることで説明しやすくなる場合があります。

ただし、増やした瞬間に良く見えるからといって、それが本当に良いモデルとは限りません。


In [ ]:
x2 = [xi ** 2 for xi in x]
feature = list(zip(x, x2))
print('feature sample =', feature[:3])
scaled_x = [(xi - x_bar) / (max(x) - min(x)) for xi in x]
print('scaled_x =', [round(v, 4) for v in scaled_x])

特徴量を増やすと、訓練データには合わせやすくなります。その代わり、偶然の揺れまで拾ってしまうと新しいデータで崩れます。ここが「表現力」と「過学習」の分かれ目です。


## 最後に、学習用と評価用を分ける

単回帰の題材でも、最後にここを入れておくことが重要です。訓練データだけで誤差を見ていると、モデルはどうしても良く見えます。評価データを別にするのは、「まだ見ていないデータでも当たるか」を確かめるためです。


In [ ]:
train_x, test_x = x[:3], x[3:]
train_y, test_y = y[:3], y[3:]
pred_test = [w0 + w1 * xi for xi in test_x]
mae_test = sum(abs(yi - pi) for yi, pi in zip(test_y, pred_test)) / len(test_y)
print('test mae =', round(mae_test, 6))

評価データで急に誤差が悪くなるなら、モデル以前にデータ数や分布差を疑うべきです。実務では、この視点を持っているかどうかでモデル改善の方向がかなり変わります。


## 振り返り

単回帰で覚えるべきことは、直線の式そのものよりも、予測を作る、外れを測る、評価用データで見直す、という流れです。

この流れが見えていれば、後でより複雑な回帰やニューラルネットワークへ進んでも、何をしているのかを見失いにくくなります。
